# Stage 6d — Structure of the largest connected component (post hoc; supports the "not one lineage" wording)

Describes the largest weakly connected component of the primary (Rule 2 v2) predicate graph:

* **Roots:** devices in the component with no in-corpus predicate; **reach** = number of devices descending
  from a root (share of the component).
* **Cross-code edges:** predicate and citing device in different FDA product codes.
* **Product-code families:** the component re-split using same-product-code edges only.
* **Body-region edges:** product codes mapped to a region from their FDA regulation number in the frozen
  510(k) snapshot (hip = 21 CFR 888.3300–888.3390; spine = 888.3050–888.3090). Lists direct hip↔spine edges
  for hand audit.

Writes `data/component_roots_by_reach.csv`, `data/component_families.csv`, `data/hip_spine_edges.csv`
and `data/component_summary.json`. Network: none. Run after Stage 6.

In [1]:
import gzip, json, collections
import pandas as pd
import networkx as nx

corpus = pd.read_csv("data/corpus.csv", dtype=str)
code = dict(zip(corpus["k_number"], corpus["product_codes"]))
info = corpus.set_index("k_number")[["device_name", "applicant", "decision_date", "product_codes"]]
edges = pd.read_csv("data/predicate_edges.csv", dtype=str)
EXPECTED = json.load(open("data/expected_values.json"))
recalled = set(pd.read_csv("data/recalled_nodes.csv", dtype=str)["k_number"])

G = nx.DiGraph(); G.add_nodes_from(corpus["k_number"])
G.add_edges_from((a, b) for a, b in zip(edges["predicate_knumber"], edges["device_knumber"]) if a in G and b in G)
H = G.subgraph(max(nx.weakly_connected_components(G), key=len)).copy()
assert H.number_of_nodes() == EXPECTED["largest_component"], "component size differs from Stage 6"
print(f"CHECKPOINT  largest component {H.number_of_nodes()} devices | {H.number_of_edges()} edges "
      f"| {len({code[n] for n in H})} product codes")

CHECKPOINT  largest component 11588 devices | 28322 edges | 111 product codes


### Roots and their reach

In [2]:
roots = [n for n in H if H.in_degree(n) == 0]
desc = {r: nx.descendants(H, r) for r in roots}
owner = collections.Counter(d for r in roots for d in desc[r])
R = pd.DataFrame([{"root": r, **info.loc[r].to_dict(), "recalled": r in recalled, "descendants": len(desc[r]),
                   "pct_of_component": round(100 * len(desc[r]) / H.number_of_nodes(), 1),
                   "only_descendant_of_this_root": sum(owner[d] == 1 for d in desc[r]), "out_degree": H.out_degree(r)}
                  for r in roots]).sort_values(["descendants", "root"], ascending=[False, True])
R.to_csv("data/component_roots_by_reach.csv", index=False)
top = R.iloc[0]
print(f"CHECKPOINT  roots {len(roots)} | top root {top['root']} ({top['device_name']}) reaches {top['descendants']} "
      f"({top['pct_of_component']}%) | roots reaching >=5%: {(R['pct_of_component'] >= 5).sum()}")
print(R.head(10)[["root", "device_name", "decision_date", "product_codes", "descendants", "pct_of_component"]].to_string(index=False))

CHECKPOINT  roots 2852 | top root K990148 (STACKABLE CAGE SYSTEM) reaches 1257 (10.8%) | roots reaching >=5%: 44
   root                             device_name decision_date product_codes  descendants  pct_of_component
K990148                   STACKABLE CAGE SYSTEM    1999-09-03           MQP         1257              10.8
K841189              OEC/REZAIAN SPINAL FIXATOR    1984-05-16           KWQ         1243              10.7
K031757              PEEK TETRIS SPINAL IMPLANT    2003-07-30           MQP         1241              10.7
K012254                                     VBR    2001-10-16           MQP         1203              10.4
K980485                     ISOLA SPINAL SYSTEM    1998-10-21           MNI         1199              10.3
K020887                    EBI AIS SPINE SYSTEM    2002-05-16           MQP         1148               9.9
K983622 BODYFORM THORACO-LUMBAR FIXATION SYSTEM    1998-12-30           KWQ         1074               9.3
K003836             SYNTHES SYN

### Cross-code edges and product-code families

In [3]:
cross = sum(code[a] != code[b] for a, b in H.edges)
S = nx.DiGraph(); S.add_nodes_from(H); S.add_edges_from((a, b) for a, b in H.edges if code[a] == code[b])
fam = sorted(nx.weakly_connected_components(S), key=len, reverse=True)
rows = []
for i, f in enumerate(fam):
    if len(f) < 10: break
    sub = S.subgraph(f); fr = [n for n in sub if sub.in_degree(n) == 0]
    best = max(fr, key=lambda r: (len(nx.descendants(sub, r)), r))
    rows.append({"family_rank": i + 1, "product_code": code[next(iter(f))], "devices": len(f), "roots": len(fr),
                 "top_root": best, "top_root_name": info.loc[best, "device_name"],
                 "top_root_reach_pct": round(100 * len(nx.descendants(sub, best)) / len(f), 1)})
F = pd.DataFrame(rows); F.to_csv("data/component_families.csv", index=False)
print(f"CHECKPOINT  cross-code edges {cross} of {H.number_of_edges()} ({100*cross/H.number_of_edges():.1f}%) | "
      f"same-code-only: largest family {len(fam[0])} | families >=10 devices {len(F)}")
print(F.head(10).to_string(index=False))

CHECKPOINT  cross-code edges 9743 of 28322 (34.4%) | same-code-only: largest family 994 | families >=10 devices 72
 family_rank product_code  devices  roots top_root                                                                                                            top_root_name  top_root_reach_pct
           1          HRS      994    235  K012655                                                                                              CONGRUENT BONE PLATE SYSTEM                26.9
           2          MAX      829    127  K071795                                                                                                  NUVASIVE COROENT SYSTEM                49.6
           3          NKB      757    123  K032033                                                                                                 CD HORIZON SPINAL SYSTEM                40.6
           4          HWC      619    164  K021932                                                                   

### Body regions from FDA regulation numbers; direct hip↔spine edges (for hand audit)

In [4]:
regc = collections.defaultdict(collections.Counter)
for r in json.load(gzip.open("snapshot/510k_raw.json.gz", "rt")):
    pc, rn = r.get("product_code"), (r.get("openfda") or {}).get("regulation_number")
    if pc and rn: regc[pc][rn if isinstance(rn, str) else rn[0]] += 1
reg = {pc: c.most_common(1)[0][0] for pc, c in regc.items()}
SPINE = {"888.3050", "888.3060", "888.3070", "888.3075", "888.3080", "888.3082", "888.3085", "888.3090"}
def region(pc):
    r = reg.get(pc, "")
    try:
        if 888.3300 <= float(r) <= 888.3390: return "hip"
    except ValueError:
        return "other"
    return "spine" if r in SPINE else "other"
E = pd.DataFrame(list(H.edges), columns=["predicate_knumber", "device_knumber"])
E["predicate_region"] = E["predicate_knumber"].map(code).map(region)
E["device_region"] = E["device_knumber"].map(code).map(region)
is_hs = ((E["predicate_region"] == "hip") & (E["device_region"] == "spine")) | \
        ((E["predicate_region"] == "spine") & (E["device_region"] == "hip"))
hs = E[is_hs]
hs.to_csv("data/hip_spine_edges.csv", index=False)
n_xreg = int((E["predicate_region"] != E["device_region"]).sum())
print(f"CHECKPOINT  cross-region edges (hip/spine/other) {n_xreg} | direct hip<->spine edges {len(hs)}")
print(hs.to_string(index=False))

CHECKPOINT  cross-region edges (hip/spine/other) 434 | direct hip<->spine edges 5
predicate_knumber device_knumber predicate_region device_region
          K000538        K063544              hip         spine
          K121297        K163625              hip         spine
          K132788        K152893            spine           hip
          K181963        K210263            spine           hip
          K953925        K030087              hip         spine


In [5]:
summary = {"devices": H.number_of_nodes(), "edges": H.number_of_edges(), "product_codes": len({code[n] for n in H}),
           "roots": len(roots), "top_root": top["root"], "top_root_reach": int(top["descendants"]),
           "top_root_pct": float(top["pct_of_component"]), "roots_reaching_5pct": int((R["pct_of_component"] >= 5).sum()),
           "cross_code_edges": int(cross), "cross_code_pct": round(100 * cross / H.number_of_edges(), 1),
           "largest_same_code_family": len(fam[0]), "families_ge10": int(len(F)),
           "cross_region_edges": n_xreg, "hip_spine_edges": int(len(hs))}
json.dump(summary, open("data/component_summary.json", "w"), indent=2)
print("CHECKPOINT  wrote data/component_summary.json", summary)

CHECKPOINT  wrote data/component_summary.json {'devices': 11588, 'edges': 28322, 'product_codes': 111, 'roots': 2852, 'top_root': 'K990148', 'top_root_reach': 1257, 'top_root_pct': 10.8, 'roots_reaching_5pct': 44, 'cross_code_edges': 9743, 'cross_code_pct': 34.4, 'largest_same_code_family': 994, 'families_ge10': 72, 'cross_region_edges': 434, 'hip_spine_edges': 5}
